# Subject Signal Diagnostics for Lee2019 SSVEP

This notebook recreates the diagnostics workflow for identifying weak downstream SSVEP subjects, with a primary focus on subject 51 and additional checks for subject 54 against stronger subjects 50 and 53.

It follows the same conventions as the fine-tuning notebook: numbered sections, config-driven execution, deterministic run IDs, structured logging, and artifact-first outputs.

# 1. Setup and Imports

Load dependencies, configure plotting defaults, and print concise runtime information.

In [ ]:
import os
import sys
import json
import hashlib
import random
import platform
from datetime import datetime
from pathlib import Path
from collections import defaultdict

import builtins
import warnings
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.signal import welch

import mne
from moabb.datasets import Lee2019_SSVEP
from braindecode.datasets import MOABBDataset
from braindecode.preprocessing import Preprocessor, preprocess, create_windows_from_events

warnings.filterwarnings("ignore")
mne.set_log_level("WARNING")

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_context("notebook")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

print("All imports loaded successfully.")
print(f"Python:   {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")

# 2. Configuration

Use a single top-level CONFIG dictionary to control all diagnostics behavior and artifact output paths.

In [ ]:
WORKING_DIR = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()

CONFIG = {
    "artifact_dir": str(WORKING_DIR / "artifacts" / "subject-signal-diagnostics"),
    "seed": 1,
    "set_seed": True,
    "paradigm": "SSVEP",
    "target_subject": 51,
    "comparison_subjects": [50, 53],
    "extra_subjects": [54],
    "inspect_runs_per_subject": 2,
    "inspection_mode": "both",  # raw, preprocessed, both
    "set_eeg_reference": True,
    "convert_to_uV": True,
    "bandpass_low": 0.5,
    "bandpass_high": 40.0,
    "sfreq": 128,
    "requested_trial_duration_s": 4.19,
    "save_artifacts": True,
    "save_figures": True,
    "save_tables": True,
    "save_logs": True,
    "verbose": True,
    "channels_to_plot": ["O1", "Oz", "O2", "PO3", "POz", "PO4", "Pz", "Cz"],
    "plot_duration_s": 8.0,
    "top_k_suspicious": 12,
    "psd_n_fft": 512,
    "harmonic_count": 3,
    "figure_dpi": 140,
    "near_flat_threshold_uV": 0.5,
    "hf_ratio_split_hz": 20.0,
    "stimulus_frequencies_hz": [5.45, 6.67, 8.57, 12.0],
    "snr_band_half_width_hz": 0.2,
    "snr_noise_exclusion_hz": 0.4,
    "snr_noise_window_hz": 1.0,
}

print("=" * 70)
print("CONFIGURATION")
print("=" * 70)
for key in sorted(CONFIG.keys()):
    print(f"  {key}: {CONFIG[key]}")
print("=" * 70)

# 3. Reproducibility, Run ID, and Artifact Paths

Set deterministic seeding, initialize run directories, and mirror the fine-tuning notebook artifact conventions.

In [ ]:
def set_global_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)


def create_run_id() -> str:
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    config_hash = hashlib.md5(json.dumps(CONFIG, sort_keys=True, default=str).encode()).hexdigest()[:8]
    return f"{timestamp}_{config_hash}"


if CONFIG["set_seed"]:
    set_global_seed(int(CONFIG["seed"]))

RUN_ID = create_run_id()
ARTIFACT_DIR = Path(CONFIG["artifact_dir"]) / CONFIG["paradigm"] / RUN_ID
FIG_DIR = ARTIFACT_DIR / "figures"
TABLE_DIR = ARTIFACT_DIR / "tables"
LOG_DIR = ARTIFACT_DIR / "logs"

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

LOG_PATH = LOG_DIR / "run.log"
_LOG_FH = open(LOG_PATH, "a", buffering=1) if CONFIG["save_logs"] else None
_ORIGINAL_PRINT = builtins.print


def _timestamped_print(*args, **kwargs):
    sep = kwargs.pop("sep", " ")
    end = kwargs.pop("end", "\n")
    flush = kwargs.pop("flush", False)
    file = kwargs.pop("file", None)

    message = sep.join(str(a) for a in args)
    leading_newlines = len(message) - len(message.lstrip("\n"))
    message_body = message[leading_newlines:]

    def _write_target(text):
        if file is None:
            sys.__stdout__.write(text)
            if flush:
                sys.__stdout__.flush()
        else:
            file.write(text)
            if flush and hasattr(file, "flush"):
                file.flush()

    if leading_newlines:
        blanks = "\n" * leading_newlines
        _write_target(blanks)
        if _LOG_FH is not None:
            _LOG_FH.write(blanks)

    if message_body:
        ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        stamped = f"[{ts}] {message_body}"
        _write_target(stamped + end)
        if _LOG_FH is not None:
            _LOG_FH.write(stamped + end)
    else:
        _write_target(end)
        if _LOG_FH is not None:
            _LOG_FH.write(end)


if CONFIG["save_logs"]:
    builtins.print = _timestamped_print

print("=" * 70)
print("RUN INITIALIZED")
print("=" * 70)
print(f"Run ID:       {RUN_ID}")
print(f"Artifact dir: {ARTIFACT_DIR}")
print(f"Figures dir:  {FIG_DIR}")
print(f"Tables dir:   {TABLE_DIR}")
print(f"Logs dir:     {LOG_DIR}")

# 4. Utilities

Define reusable helpers for logging banners, artifact I/O, channel metrics, suspiciousness scoring, and plotting.

In [ ]:
ARTIFACT_MANIFEST: List[Dict[str, str]] = []


def add_manifest_entry(path: Path, kind: str):
    rel = path.relative_to(ARTIFACT_DIR)
    ARTIFACT_MANIFEST.append({"type": kind, "path": str(rel)})


def banner(title: str):
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)


def save_json(data: dict, path: Path, enabled: bool = True):
    if not enabled:
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w") as f:
        json.dump(data, f, indent=2, default=str)
    add_manifest_entry(path, "json")
    print(f"Saved JSON: {path}")


def save_table(df: pd.DataFrame, path: Path, enabled: bool = True):
    if not enabled:
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False)
    add_manifest_entry(path, "table")
    print(f"Saved table: {path}")


def save_figure(fig: plt.Figure, path: Path, enabled: bool = True):
    if enabled:
        path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(path, dpi=CONFIG["figure_dpi"], bbox_inches="tight")
        add_manifest_entry(path, "figure")
        print(f"Saved figure: {path}")
    plt.close(fig)


def robust_mad(x: np.ndarray, axis: int = -1) -> np.ndarray:
    med = np.median(x, axis=axis, keepdims=True)
    return np.median(np.abs(x - med), axis=axis)


def line_length(x: np.ndarray) -> np.ndarray:
    return np.mean(np.abs(np.diff(x, axis=-1)), axis=-1)


def near_flat_ratio(x: np.ndarray, threshold_uV: float) -> np.ndarray:
    threshold = threshold_uV if np.nanmedian(np.abs(x)) > 1e-3 else threshold_uV * 1e-6
    return np.mean(np.abs(x) < threshold, axis=-1)


def hf_power_ratio(x: np.ndarray, sfreq: float, split_hz: float = 20.0) -> np.ndarray:
    n_fft = min(int(CONFIG["psd_n_fft"]), x.shape[-1])
    freqs, psd = welch(x, fs=sfreq, nperseg=n_fft, axis=-1)
    low_mask = (freqs >= 1.0) & (freqs < split_hz)
    high_mask = (freqs >= split_hz) & (freqs <= min(CONFIG["bandpass_high"], sfreq / 2.0))

    # NumPy 2.x removed np.trapz; prefer trapezoid while keeping backward compatibility.
    integrate = np.trapezoid if hasattr(np, "trapezoid") else np.trapz
    low_power = integrate(psd[..., low_mask], freqs[low_mask], axis=-1)
    high_power = integrate(psd[..., high_mask], freqs[high_mask], axis=-1)
    return high_power / np.maximum(low_power, 1e-12)


def compute_channel_metrics(raw: mne.io.BaseRaw, subject: int, run_rank: int, mode: str) -> pd.DataFrame:
    data = raw.get_data(picks="eeg")
    sfreq = float(raw.info["sfreq"])
    ch_names = raw.copy().pick("eeg").ch_names

    std = np.std(data, axis=1)
    mad = robust_mad(data, axis=1)
    ptp = np.ptp(data, axis=1)
    ll = line_length(data)
    flat = near_flat_ratio(data, threshold_uV=CONFIG["near_flat_threshold_uV"])
    hf = hf_power_ratio(data, sfreq=sfreq, split_hz=float(CONFIG["hf_ratio_split_hz"]))

    return pd.DataFrame(
        {
            "subject": subject,
            "run_rank": run_rank,
            "mode": mode,
            "channel": ch_names,
            "std": std,
            "mad": mad,
            "ptp": ptp,
            "line_length": ll,
            "near_flat_ratio": flat,
            "hf_ratio": hf,
        }
    )


def apply_preprocessing(raw: mne.io.BaseRaw) -> mne.io.BaseRaw:
    out = raw.copy().load_data()
    if CONFIG["set_eeg_reference"]:
        out.set_eeg_reference("average")
    if CONFIG["convert_to_uV"]:
        out.apply_function(lambda x: x * 1e6, picks="eeg")
    out.filter(l_freq=float(CONFIG["bandpass_low"]), h_freq=float(CONFIG["bandpass_high"]), picks="eeg")
    out.resample(float(CONFIG["sfreq"]))
    return out


def psd_mean_curve(raw: mne.io.BaseRaw, picks: List[str]) -> Tuple[np.ndarray, np.ndarray]:
    keep = [c for c in picks if c in raw.ch_names]
    if not keep:
        keep = raw.copy().pick("eeg").ch_names[:8]
    data = raw.get_data(picks=keep)
    n_fft = min(int(CONFIG["psd_n_fft"]), data.shape[-1])
    freqs, psd = welch(np.mean(data, axis=0), fs=float(raw.info["sfreq"]), nperseg=n_fft)
    return freqs, psd


def suspicious_channel_scores(target_df: pd.DataFrame, comparison_df: pd.DataFrame) -> pd.DataFrame:
    metrics = ["std", "mad", "ptp", "line_length", "near_flat_ratio", "hf_ratio"]
    comp_stats = comparison_df.groupby("channel")[metrics].agg(["mean", "std"])
    rows = []
    for _, row in target_df.iterrows():
        ch = row["channel"]
        if ch not in comp_stats.index:
            continue
        z_values = {}
        for m in metrics:
            mu = comp_stats.loc[ch, (m, "mean")]
            sd = max(comp_stats.loc[ch, (m, "std")], 1e-12)
            z_values[f"z_{m}"] = (row[m] - mu) / sd
        score = float(np.mean([abs(v) for v in z_values.values()]))
        rows.append({"channel": ch, "suspicious_score": score, **z_values})
    out = pd.DataFrame(rows).sort_values("suspicious_score", ascending=False)
    return out


runtime_metadata = {
    "run_id": RUN_ID,
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "working_dir": str(WORKING_DIR),
    "artifact_dir": str(ARTIFACT_DIR),
    "python_version": sys.version,
    "platform": platform.platform(),
    "inspection_mode": CONFIG["inspection_mode"],
}

save_json(CONFIG, ARTIFACT_DIR / "config.json", enabled=CONFIG["save_artifacts"])
save_json(runtime_metadata, ARTIFACT_DIR / "run_metadata.json", enabled=CONFIG["save_artifacts"])

print("Utility initialization complete.")

# 5. Data Loading

Load Lee2019 SSVEP through MOABB and build a tidy per-recording index for subject, run/session, and basic signal properties.

In [ ]:
banner("DATA LOADING")

subject_ids = sorted(
    set([CONFIG["target_subject"]] + CONFIG["comparison_subjects"] + CONFIG["extra_subjects"])
)

base_dataset = Lee2019_SSVEP()
print(f"MOABB dataset code: {base_dataset.code}")
print(f"Event mapping: {base_dataset.event_id}")
print(f"Selected subjects: {subject_ids}")

DATASET_NAME = f"Lee2019_{CONFIG['paradigm']}"
DATASET = MOABBDataset(dataset_name=DATASET_NAME, subject_ids=subject_ids)
print(f"Loaded recordings: {len(DATASET.datasets)}")

recordings: List[Dict] = []
subject_run_counter = defaultdict(int)

for i, ds in enumerate(DATASET.datasets):
    raw = ds.raw
    desc = dict(ds.description)

    subject = int(desc.get("subject", -1))
    subject_run_counter[subject] += 1
    run_rank = subject_run_counter[subject]

    rec = {
        "recording_id": i,
        "subject": subject,
        "session": str(desc.get("session", f"session-{run_rank}")),
        "run": str(desc.get("run", f"run-{run_rank}")),
        "run_rank": run_rank,
        "sfreq": float(raw.info["sfreq"]),
        "n_channels": len(raw.copy().pick("eeg").ch_names),
        "duration_s": float(raw.n_times / raw.info["sfreq"]),
        "raw": raw,
        "description": desc,
    }
    recordings.append(rec)

recording_index_df = pd.DataFrame(
    [
        {
            "recording_id": r["recording_id"],
            "subject": r["subject"],
            "session": r["session"],
            "run": r["run"],
            "run_rank": r["run_rank"],
            "sfreq": r["sfreq"],
            "n_channels": r["n_channels"],
            "duration_s": r["duration_s"],
        }
        for r in recordings
    ]
).sort_values(["subject", "run_rank"])

print("\nRecording index summary:")
print(recording_index_df)

print("\nChannel count check:")
print(recording_index_df.groupby("subject")["n_channels"].unique())

save_table(recording_index_df, TABLE_DIR / "recording_index.csv", enabled=CONFIG["save_tables"])

recording_overview = {
    "subjects": subject_ids,
    "n_recordings": int(len(recordings)),
    "per_subject_recordings": recording_index_df.groupby("subject").size().to_dict(),
    "event_id": base_dataset.event_id,
}
save_json(recording_overview, ARTIFACT_DIR / "recording_overview.json", enabled=CONFIG["save_artifacts"])

# 6. Preprocessing

Build raw and preprocessed views that mirror downstream assumptions: average reference, uV conversion, bandpass 0.5-40.0 Hz, and resample to 128 Hz.

In [ ]:
banner("PREPROCESSING")

view_recordings: Dict[str, List[Dict]] = {"raw": recordings}

if CONFIG["inspection_mode"] in {"preprocessed", "both"}:
    preprocessed = []
    for rec in recordings:
        pre_rec = dict(rec)
        pre_rec["raw"] = apply_preprocessing(rec["raw"])
        preprocessed.append(pre_rec)
    view_recordings["preprocessed"] = preprocessed

if CONFIG["inspection_mode"] == "preprocessed":
    view_recordings = {"preprocessed": view_recordings["preprocessed"]}

if CONFIG["inspection_mode"] == "both":
    view_recordings = {"raw": recordings, "preprocessed": view_recordings["preprocessed"]}

preproc_rows = []
for mode, recs in view_recordings.items():
    for rec in recs:
        preproc_rows.append(
            {
                "mode": mode,
                "subject": rec["subject"],
                "run_rank": rec["run_rank"],
                "sfreq": float(rec["raw"].info["sfreq"]),
                "n_channels": len(rec["raw"].copy().pick("eeg").ch_names),
                "duration_s": float(rec["raw"].n_times / rec["raw"].info["sfreq"]),
            }
        )

preprocessing_summary_df = pd.DataFrame(preproc_rows).sort_values(["mode", "subject", "run_rank"])
print(preprocessing_summary_df)
save_table(preprocessing_summary_df, TABLE_DIR / "preprocessing_summary.csv", enabled=CONFIG["save_tables"])

preprocessing_config = {
    "set_eeg_reference": CONFIG["set_eeg_reference"],
    "convert_to_uV": CONFIG["convert_to_uV"],
    "bandpass_low": CONFIG["bandpass_low"],
    "bandpass_high": CONFIG["bandpass_high"],
    "sfreq": CONFIG["sfreq"],
    "inspection_mode": CONFIG["inspection_mode"],
}
save_json(preprocessing_config, ARTIFACT_DIR / "preprocessing_config.json", enabled=CONFIG["save_artifacts"])

# 7. Continuous-Signal Diagnostics

Compute channel and run metrics for continuous recordings and visualize subject-channel structure across raw and preprocessed modes.

In [ ]:
banner("CONTINUOUS SIGNAL DIAGNOSTICS")

channel_metrics_frames = []
for mode, recs in view_recordings.items():
    for rec in recs:
        cm = compute_channel_metrics(rec["raw"], rec["subject"], rec["run_rank"], mode)
        channel_metrics_frames.append(cm)

channel_metrics_df = pd.concat(channel_metrics_frames, ignore_index=True)
run_metrics_df = (
    channel_metrics_df
    .groupby(["mode", "subject", "run_rank"], as_index=False)[["std", "mad", "ptp", "line_length", "near_flat_ratio", "hf_ratio"]]
    .mean()
)
subject_metrics_df = (
    channel_metrics_df
    .groupby(["mode", "subject"], as_index=False)[["std", "mad", "ptp", "line_length", "near_flat_ratio", "hf_ratio"]]
    .mean()
)

print("Run-level metrics:")
print(run_metrics_df)

save_table(channel_metrics_df, TABLE_DIR / "channel_metrics.csv", enabled=CONFIG["save_tables"])
save_table(run_metrics_df, TABLE_DIR / "run_metrics.csv", enabled=CONFIG["save_tables"])
save_table(subject_metrics_df, TABLE_DIR / "subject_metrics.csv", enabled=CONFIG["save_tables"])

mode_for_plots = "preprocessed" if "preprocessed" in view_recordings else list(view_recordings.keys())[0]
plot_recs = view_recordings[mode_for_plots]

# Stacked traces for selected channels.
for rec in plot_recs:
    if rec["subject"] not in [CONFIG["target_subject"], *CONFIG["comparison_subjects"], *CONFIG["extra_subjects"]]:
        continue
    raw = rec["raw"]
    sfreq = raw.info["sfreq"]
    n_times = int(min(raw.n_times, CONFIG["plot_duration_s"] * sfreq))
    picks = [c for c in CONFIG["channels_to_plot"] if c in raw.ch_names]
    if not picks:
        picks = raw.copy().pick("eeg").ch_names[:8]
    data = raw.get_data(picks=picks, start=0, stop=n_times)
    t = np.arange(data.shape[1]) / sfreq

    fig, ax = plt.subplots(figsize=(11, 5))
    offset = np.nanmedian(np.std(data, axis=1)) * 6.0
    for i, ch in enumerate(picks):
        ax.plot(t, data[i] + i * offset, lw=0.9, label=ch)
    ax.set_title(f"{mode_for_plots} stacked traces | subject {rec['subject']} run {rec['run_rank']}")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Amplitude + offset")
    ax.legend(ncol=4, fontsize=8, loc="upper right")
    save_figure(
        fig,
        FIG_DIR / f"stacked_traces_{mode_for_plots}_sub{rec['subject']}_run{rec['run_rank']}.png",
        enabled=CONFIG["save_figures"],
    )

# Heatmap: channel std by subject and run.
std_slice = channel_metrics_df[channel_metrics_df["mode"] == mode_for_plots].copy()
std_slice["subject_run"] = std_slice["subject"].astype(str) + "-r" + std_slice["run_rank"].astype(str)
std_pivot = std_slice.pivot_table(index="channel", columns="subject_run", values="std", aggfunc="mean")

fig, ax = plt.subplots(figsize=(10, 14))
sns.heatmap(std_pivot, cmap="mako", ax=ax)
ax.set_title(f"Channel STD heatmap ({mode_for_plots})")
save_figure(fig, FIG_DIR / f"heatmap_channel_std_{mode_for_plots}.png", enabled=CONFIG["save_figures"])

# Heatmap: channel HF ratio by subject and run.
hf_pivot = std_slice.pivot_table(index="channel", columns="subject_run", values="hf_ratio", aggfunc="mean")
fig, ax = plt.subplots(figsize=(10, 14))
sns.heatmap(hf_pivot, cmap="rocket", ax=ax)
ax.set_title(f"Channel high-frequency power ratio heatmap ({mode_for_plots})")
save_figure(fig, FIG_DIR / f"heatmap_channel_hf_ratio_{mode_for_plots}.png", enabled=CONFIG["save_figures"])

# 8. Session / Run Diagnostics

Directly compare run 1 versus run 2 behavior per subject with metric deltas, PSD overlays, and run-specific suspicious channel ranking.

In [ ]:
banner("SESSION / RUN DIAGNOSTICS")

run_delta_rows = []
for mode in sorted(run_metrics_df["mode"].unique()):
    mode_df = run_metrics_df[run_metrics_df["mode"] == mode]
    for subject in sorted(mode_df["subject"].unique()):
        s_df = mode_df[mode_df["subject"] == subject].sort_values("run_rank")
        if len(s_df) < 2:
            continue
        r1 = s_df.iloc[0]
        r2 = s_df.iloc[1]
        run_delta_rows.append(
            {
                "mode": mode,
                "subject": subject,
                "std_delta_r2_minus_r1": r2["std"] - r1["std"],
                "mad_delta_r2_minus_r1": r2["mad"] - r1["mad"],
                "line_length_delta_r2_minus_r1": r2["line_length"] - r1["line_length"],
                "hf_ratio_delta_r2_minus_r1": r2["hf_ratio"] - r1["hf_ratio"],
            }
        )

run_delta_df = pd.DataFrame(run_delta_rows)
print("Run deltas:")
print(run_delta_df)
save_table(run_delta_df, TABLE_DIR / "run_delta_metrics.csv", enabled=CONFIG["save_tables"])

# Side-by-side PSD for each subject run1 vs run2.
mode_for_psd = "preprocessed" if "preprocessed" in view_recordings else "raw"
for subject in subject_ids:
    recs = [r for r in view_recordings[mode_for_psd] if r["subject"] == subject]
    recs = sorted(recs, key=lambda x: x["run_rank"])[:2]
    if len(recs) < 2:
        continue

    fig, ax = plt.subplots(figsize=(10, 4))
    for rec in recs:
        freqs, psd = psd_mean_curve(rec["raw"], picks=CONFIG["channels_to_plot"])
        ax.plot(freqs, psd, lw=1.5, label=f"run {rec['run_rank']}")
    ax.set_xlim(1, CONFIG["bandpass_high"])
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("PSD")
    ax.set_title(f"Subject {subject} run-wise PSD ({mode_for_psd})")
    ax.legend()
    save_figure(fig, FIG_DIR / f"run_psd_subject_{subject}_{mode_for_psd}.png", enabled=CONFIG["save_figures"])

# Per-run suspicious channels for target subject.
per_run_suspicious_frames = []
comparison_subjects = set(CONFIG["comparison_subjects"])
for run_rank in sorted(channel_metrics_df["run_rank"].unique()):
    target_run = channel_metrics_df[
        (channel_metrics_df["subject"] == CONFIG["target_subject"]) &
        (channel_metrics_df["run_rank"] == run_rank) &
        (channel_metrics_df["mode"] == mode_for_psd)
    ]
    comp_run = channel_metrics_df[
        (channel_metrics_df["subject"].isin(comparison_subjects)) &
        (channel_metrics_df["run_rank"] == run_rank) &
        (channel_metrics_df["mode"] == mode_for_psd)
    ]
    if len(target_run) == 0 or len(comp_run) == 0:
        continue

    s = suspicious_channel_scores(target_run, comp_run)
    s["run_rank"] = run_rank
    per_run_suspicious_frames.append(s)

run_suspicious_df = pd.concat(per_run_suspicious_frames, ignore_index=True) if per_run_suspicious_frames else pd.DataFrame()
if len(run_suspicious_df) > 0:
    save_table(run_suspicious_df, TABLE_DIR / "run_suspicious_channels.csv", enabled=CONFIG["save_tables"])

    top_plot = run_suspicious_df.sort_values(["run_rank", "suspicious_score"], ascending=[True, False]).groupby("run_rank").head(8)
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.barplot(data=top_plot, x="channel", y="suspicious_score", hue="run_rank", ax=ax)
    ax.set_title("Target subject suspicious channels by run")
    ax.tick_params(axis="x", rotation=45)
    save_figure(fig, FIG_DIR / "target_run_suspicious_channels.png", enabled=CONFIG["save_figures"])
else:
    print("No per-run suspicious channel table generated.")

# 9. Window-Level Diagnostics

Construct downstream-style windows using the configured duration and analyze per-window quality, variability, and outliers.

In [ ]:
banner("WINDOW-LEVEL DIAGNOSTICS")

WINDOW_DATASET = MOABBDataset(dataset_name=DATASET_NAME, subject_ids=subject_ids)

window_preprocessors = [Preprocessor("pick_types", eeg=True, meg=False, stim=False)]
if CONFIG["set_eeg_reference"]:
    window_preprocessors.append(Preprocessor("set_eeg_reference", ref_channels="average", ch_type="eeg"))
if CONFIG["convert_to_uV"]:
    window_preprocessors.append(Preprocessor(lambda x: x * 1e6, apply_on_array=True))
window_preprocessors.extend(
    [
        Preprocessor("filter", l_freq=float(CONFIG["bandpass_low"]), h_freq=float(CONFIG["bandpass_high"])),
        Preprocessor("resample", sfreq=float(CONFIG["sfreq"])),
    ]
)

preprocess(WINDOW_DATASET, window_preprocessors, n_jobs=1)

target_sfreq = float(CONFIG["sfreq"])
base_trial_duration_s = 4.0
window_samples = int(round(CONFIG["requested_trial_duration_s"] * target_sfreq))
base_trial_samples = int(round(base_trial_duration_s * target_sfreq))
trial_stop_offset_samples = window_samples - base_trial_samples

label_map = {k: i for i, k in enumerate(sorted(base_dataset.event_id.keys()))}

WINDOWS_DATASET = create_windows_from_events(
    WINDOW_DATASET,
    trial_start_offset_samples=0,
    trial_stop_offset_samples=trial_stop_offset_samples,
    window_size_samples=None,
    window_stride_samples=None,
    drop_last_window=False,
    mapping=label_map,
    preload=True,
    on_missing="ignore",
)

window_rows = []
window_shape_reference = None
for ds in WINDOWS_DATASET.datasets:
    sid = int(ds.description.get("subject", -1))
    run_rank = int(ds.description.get("run", ds.description.get("session", 1))) if str(ds.description.get("run", "")).isdigit() else 1

    X = ds.windows.get_data(copy=False)
    y = ds.y

    if window_shape_reference is None and len(X) > 0:
        window_shape_reference = tuple(X.shape[1:])

    energy = np.mean(X ** 2, axis=(1, 2))
    variance = np.var(X, axis=(1, 2))
    ptp = np.mean(np.ptp(X, axis=2), axis=1)
    ll = np.mean(np.mean(np.abs(np.diff(X, axis=2)), axis=2), axis=1)

    for i in range(len(X)):
        window_rows.append(
            {
                "subject": sid,
                "run_rank": run_rank,
                "class": int(y[i]),
                "window_idx": i,
                "energy": float(energy[i]),
                "variance": float(variance[i]),
                "ptp": float(ptp[i]),
                "line_length": float(ll[i]),
            }
        )

window_metrics_df = pd.DataFrame(window_rows)
window_metrics_df["group"] = np.where(
    window_metrics_df["subject"] == CONFIG["target_subject"],
    "target",
    np.where(window_metrics_df["subject"].isin(CONFIG["comparison_subjects"]), "comparison", "extra"),
)

for col in ["energy", "variance", "line_length"]:
    med = window_metrics_df[col].median()
    mad = np.median(np.abs(window_metrics_df[col] - med)) + 1e-12
    window_metrics_df[f"rz_{col}"] = 0.6745 * (window_metrics_df[col] - med) / mad

window_metrics_df["outlier_flag"] = (
    (window_metrics_df["rz_energy"].abs() > 3.5)
    | (window_metrics_df["rz_variance"].abs() > 3.5)
    | (window_metrics_df["rz_line_length"].abs() > 3.5)
)

print(f"Final window shape reference (n_channels, n_times): {window_shape_reference}")
print(f"Total windows: {len(window_metrics_df)}")
print(window_metrics_df.groupby(["subject", "class"]).size().unstack(fill_value=0))

save_table(window_metrics_df, TABLE_DIR / "window_metrics.csv", enabled=CONFIG["save_tables"])
save_table(window_metrics_df[window_metrics_df["outlier_flag"]], TABLE_DIR / "outlier_windows.csv", enabled=CONFIG["save_tables"])

# Compare target vs comparison distributions.
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, ["energy", "variance", "line_length"]):
    sns.boxplot(data=window_metrics_df[window_metrics_df["group"].isin(["target", "comparison"])], x="group", y=col, ax=ax)
    ax.set_title(f"Window {col}")
save_figure(fig, FIG_DIR / "window_metric_target_vs_comparison.png", enabled=CONFIG["save_figures"])

# Representative window traces and channel x time heatmaps.
def collect_subject_windows(subject: int, max_windows: int = 30):
    chunks = []
    for ds in WINDOWS_DATASET.datasets:
        sid = int(ds.description.get("subject", -1))
        if sid != subject:
            continue
        X = ds.windows.get_data(copy=False)
        if len(X) == 0:
            continue
        chunks.append(X[:max_windows])
    if not chunks:
        return None
    return np.concatenate(chunks, axis=0)

X_target = collect_subject_windows(CONFIG["target_subject"])
X_comp = np.concatenate([w for s in CONFIG["comparison_subjects"] if (w := collect_subject_windows(s)) is not None], axis=0) if any(collect_subject_windows(s) is not None for s in CONFIG["comparison_subjects"]) else None

if X_target is not None and X_comp is not None:
    target_mean = np.mean(X_target, axis=0)
    comp_mean = np.mean(X_comp, axis=0)

    ch_idx = 0
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(target_mean[ch_idx], label=f"target s{CONFIG['target_subject']}")
    ax.plot(comp_mean[ch_idx], label="comparison mean")
    ax.set_title("Representative window trace (channel index 0)")
    ax.set_xlabel("Time samples")
    ax.set_ylabel("Amplitude")
    ax.legend()
    save_figure(fig, FIG_DIR / "representative_window_trace.png", enabled=CONFIG["save_figures"])

    fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)
    sns.heatmap(np.abs(target_mean), ax=axes[0], cmap="viridis")
    axes[0].set_title("Target mean |window| (channel x time)")
    axes[0].set_xlabel("Time")
    axes[0].set_ylabel("Channel")

    sns.heatmap(np.abs(comp_mean), ax=axes[1], cmap="viridis")
    axes[1].set_title("Comparison mean |window| (channel x time)")
    axes[1].set_xlabel("Time")
    save_figure(fig, FIG_DIR / "window_channel_time_heatmaps.png", enabled=CONFIG["save_figures"])
else:
    print("Insufficient windows for target/comparison representative plots.")

# 10. Frequency / Class Diagnostics

Evaluate class-aware SSVEP spectral structure with stimulus frequencies and harmonics marked, and compute simple signal-to-background ratios.

In [ ]:
banner("FREQUENCY / CLASS DIAGNOSTICS")

class_metrics_rows = []

# Collect windows grouped by class and subject group.
def windows_by_group_and_class(group_subjects: List[int]):
    out = defaultdict(list)
    for ds in WINDOWS_DATASET.datasets:
        sid = int(ds.description.get("subject", -1))
        if sid not in group_subjects:
            continue
        X = ds.windows.get_data(copy=False)
        y = ds.y
        for cls in np.unique(y):
            out[int(cls)].append(X[y == cls])
    return {k: np.concatenate(v, axis=0) for k, v in out.items() if len(v) > 0}


target_cls = windows_by_group_and_class([CONFIG["target_subject"]])
comp_cls = windows_by_group_and_class(CONFIG["comparison_subjects"])

freq_axis_ref = None
for cls in sorted(set(list(target_cls.keys()) + list(comp_cls.keys()))):
    fig, ax = plt.subplots(figsize=(10, 4))

    for group_name, grouped in [("target", target_cls), ("comparison", comp_cls)]:
        if cls not in grouped:
            continue
        X = grouped[cls]
        signal = np.mean(X, axis=1)  # average channels per window
        freqs, psd = welch(signal, fs=float(CONFIG["sfreq"]), nperseg=min(CONFIG["psd_n_fft"], X.shape[-1]), axis=-1)
        mean_psd = np.mean(psd, axis=0)
        ax.plot(freqs, mean_psd, lw=1.6, label=f"{group_name} class {cls}")

        if freq_axis_ref is None:
            freq_axis_ref = freqs

        stim_freq = CONFIG["stimulus_frequencies_hz"][cls] if cls < len(CONFIG["stimulus_frequencies_hz"]) else None
        if stim_freq is not None:
            band = float(CONFIG["snr_band_half_width_hz"])
            noise_win = float(CONFIG["snr_noise_window_hz"])
            excl = float(CONFIG["snr_noise_exclusion_hz"])

            sig_mask = (freqs >= stim_freq - band) & (freqs <= stim_freq + band)
            noise_mask = (freqs >= stim_freq - noise_win) & (freqs <= stim_freq + noise_win) & (~((freqs >= stim_freq - excl) & (freqs <= stim_freq + excl)))

            sig_power = float(np.mean(mean_psd[sig_mask])) if np.any(sig_mask) else np.nan
            noise_power = float(np.mean(mean_psd[noise_mask])) if np.any(noise_mask) else np.nan
            snr = sig_power / max(noise_power, 1e-12) if np.isfinite(noise_power) else np.nan

            class_metrics_rows.append(
                {
                    "group": group_name,
                    "class": cls,
                    "stimulus_frequency_hz": stim_freq,
                    "signal_power": sig_power,
                    "background_power": noise_power,
                    "snr": snr,
                }
            )

    for stim_f in CONFIG["stimulus_frequencies_hz"]:
        for h in range(1, int(CONFIG["harmonic_count"]) + 1):
            f = stim_f * h
            if f <= CONFIG["bandpass_high"]:
                ax.axvline(f, color="gray", alpha=0.18, lw=0.8)

    ax.set_xlim(1, CONFIG["bandpass_high"])
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("PSD")
    ax.set_title(f"Class-wise PSD with stimulus harmonics (class {cls})")
    ax.legend()
    save_figure(fig, FIG_DIR / f"class_psd_{cls}.png", enabled=CONFIG["save_figures"])

class_frequency_metrics_df = pd.DataFrame(class_metrics_rows)
print(class_frequency_metrics_df)
save_table(class_frequency_metrics_df, TABLE_DIR / "class_frequency_metrics.csv", enabled=CONFIG["save_tables"])

if len(class_frequency_metrics_df) > 0:
    fig, ax = plt.subplots(figsize=(9, 4))
    sns.barplot(data=class_frequency_metrics_df, x="class", y="snr", hue="group", ax=ax)
    ax.set_title("Class-wise SSVEP signal-to-background ratio")
    save_figure(fig, FIG_DIR / "class_snr_comparison.png", enabled=CONFIG["save_figures"])

# 11. Suspicious Channel Analysis and Spatial Diagnostics

Rank target-subject channels by z-scored deviations from comparison subjects and render topomap-style views when montage information is available.

In [ ]:
banner("SUSPICIOUS CHANNEL ANALYSIS")

primary_mode = "preprocessed" if "preprocessed" in channel_metrics_df["mode"].unique() else "raw"

target_channel = (
    channel_metrics_df[
        (channel_metrics_df["subject"] == CONFIG["target_subject"]) &
        (channel_metrics_df["mode"] == primary_mode)
    ]
    .groupby("channel", as_index=False)[["std", "mad", "ptp", "line_length", "near_flat_ratio", "hf_ratio"]]
    .mean()
)
comparison_channel = (
    channel_metrics_df[
        (channel_metrics_df["subject"].isin(CONFIG["comparison_subjects"])) &
        (channel_metrics_df["mode"] == primary_mode)
    ]
    .groupby(["subject", "channel"], as_index=False)[["std", "mad", "ptp", "line_length", "near_flat_ratio", "hf_ratio"]]
    .mean()
)

suspicious_df = suspicious_channel_scores(target_channel, comparison_channel)
top_k = int(CONFIG["top_k_suspicious"])
suspicious_top = suspicious_df.head(top_k)

print("Top suspicious channels:")
print(suspicious_top)

save_table(suspicious_df, TABLE_DIR / "suspicious_channels.csv", enabled=CONFIG["save_tables"])

fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=suspicious_top, x="channel", y="suspicious_score", color="#CC4E5C", ax=ax)
ax.set_title(f"Top {top_k} suspicious channels for target subject {CONFIG['target_subject']}")
ax.tick_params(axis="x", rotation=45)
save_figure(fig, FIG_DIR / "suspicious_channel_leaderboard.png", enabled=CONFIG["save_figures"])

# Worst-channel raw trace snapshots.
if len(suspicious_top) > 0:
    worst_channels = suspicious_top["channel"].head(4).tolist()
    target_runs = [r for r in view_recordings[primary_mode] if r["subject"] == CONFIG["target_subject"]]
    for rec in sorted(target_runs, key=lambda x: x["run_rank"])[:2]:
        raw = rec["raw"]
        sfreq = raw.info["sfreq"]
        n_times = int(min(raw.n_times, CONFIG["plot_duration_s"] * sfreq))
        data = raw.get_data(picks=[c for c in worst_channels if c in raw.ch_names], start=0, stop=n_times)
        t = np.arange(data.shape[1]) / sfreq

        fig, axes = plt.subplots(data.shape[0], 1, figsize=(10, 2.5 * max(1, data.shape[0])), sharex=True)
        if data.shape[0] == 1:
            axes = [axes]
        for i, ax in enumerate(axes):
            ax.plot(t, data[i], lw=1.0)
            ax.set_ylabel(worst_channels[i])
        axes[-1].set_xlabel("Time (s)")
        fig.suptitle(f"Worst-channel traces | target {CONFIG['target_subject']} run {rec['run_rank']}")
        save_figure(fig, FIG_DIR / f"worst_channel_traces_target_run{rec['run_rank']}.png", enabled=CONFIG["save_figures"])

# Topomap-style diagnostics with fallback to channel-bar spatial proxy.
def save_topomap_or_proxy(series: pd.Series, title: str, fname: str):
    target_rec = next((r for r in view_recordings[primary_mode] if r["subject"] == CONFIG["target_subject"]), None)
    if target_rec is None:
        return

    raw = target_rec["raw"].copy().pick("eeg")
    info = raw.info.copy()
    if info.get_montage() is None:
        info.set_montage("standard_1020", on_missing="ignore")

    keep = [ch for ch in info.ch_names if ch in series.index]
    vals = np.array([series.loc[ch] for ch in keep], dtype=float)

    try:
        pick_idx = [info.ch_names.index(ch) for ch in keep]
        info_sub = mne.pick_info(info, pick_idx)
        fig, ax = plt.subplots(figsize=(6, 5))
        mne.viz.plot_topomap(vals, info_sub, axes=ax, show=False, contours=0)
        ax.set_title(title)
        save_figure(fig, FIG_DIR / fname, enabled=CONFIG["save_figures"])
    except Exception as exc:
        fig, ax = plt.subplots(figsize=(10, 4))
        ranked = pd.Series(vals, index=keep).sort_values(ascending=False)
        ranked.plot(kind="bar", ax=ax, color="#4C78A8")
        ax.set_title(f"{title} (proxy view; topomap unavailable)")
        ax.set_ylabel("Value")
        ax.tick_params(axis="x", rotation=60)
        save_figure(fig, FIG_DIR / fname, enabled=CONFIG["save_figures"])
        print(f"Topomap fallback used: {exc}")


std_map = target_channel.set_index("channel")["std"]
score_map = suspicious_df.set_index("channel")["suspicious_score"]

save_topomap_or_proxy(std_map, "Target subject channel std", "topomap_channel_std.png")
save_topomap_or_proxy(score_map, "Target subject suspiciousness", "topomap_suspiciousness.png")

# Target-frequency spectral strength map.
target_freq = CONFIG["stimulus_frequencies_hz"][0]
spec_rows = []
for rec in [r for r in view_recordings[primary_mode] if r["subject"] == CONFIG["target_subject"]]:
    raw = rec["raw"]
    data = raw.get_data(picks="eeg")
    freqs, psd = welch(data, fs=float(raw.info["sfreq"]), nperseg=min(CONFIG["psd_n_fft"], data.shape[-1]), axis=-1)
    mask = (freqs >= target_freq - 0.3) & (freqs <= target_freq + 0.3)
    power = np.mean(psd[:, mask], axis=1)
    for ch, p in zip(raw.copy().pick("eeg").ch_names, power):
        spec_rows.append({"channel": ch, "power": float(p)})

spec_df = pd.DataFrame(spec_rows)
if len(spec_df) > 0:
    spec_map = spec_df.groupby("channel")["power"].mean()
    save_topomap_or_proxy(spec_map, f"Target spectral power near {target_freq} Hz", "topomap_target_freq_strength.png")

# 12. Summary and Artifact Manifest / Output Paths

Consolidate key findings and write summary plus manifest files for reproducibility and downstream review.

In [ ]:
banner("FINAL SUMMARY")

target_mode_df = run_metrics_df[
    (run_metrics_df["subject"] == CONFIG["target_subject"]) &
    (run_metrics_df["mode"] == primary_mode)
].sort_values("run_rank")

target_vs_comp = subject_metrics_df[subject_metrics_df["mode"] == primary_mode].copy()
comp_avg = target_vs_comp[target_vs_comp["subject"].isin(CONFIG["comparison_subjects"])][["std", "mad", "line_length", "hf_ratio"]].mean()
target_avg = target_vs_comp[target_vs_comp["subject"] == CONFIG["target_subject"]][["std", "mad", "line_length", "hf_ratio"]].mean()

summary = {
    "run_id": RUN_ID,
    "artifact_dir": str(ARTIFACT_DIR),
    "target_subject": int(CONFIG["target_subject"]),
    "comparison_subjects": [int(s) for s in CONFIG["comparison_subjects"]],
    "extra_subjects": [int(s) for s in CONFIG["extra_subjects"]],
    "inspection_mode": CONFIG["inspection_mode"],
    "preprocessing": {
        "set_eeg_reference": CONFIG["set_eeg_reference"],
        "convert_to_uV": CONFIG["convert_to_uV"],
        "bandpass_low": CONFIG["bandpass_low"],
        "bandpass_high": CONFIG["bandpass_high"],
        "sfreq": CONFIG["sfreq"],
    },
    "window_shape": window_shape_reference,
    "n_windows": int(len(window_metrics_df)),
    "n_outlier_windows": int(window_metrics_df["outlier_flag"].sum()) if len(window_metrics_df) else 0,
    "target_vs_comparison_mean_metrics": {
        "target_std": float(target_avg.get("std", np.nan)),
        "comparison_std": float(comp_avg.get("std", np.nan)),
        "target_hf_ratio": float(target_avg.get("hf_ratio", np.nan)),
        "comparison_hf_ratio": float(comp_avg.get("hf_ratio", np.nan)),
        "target_line_length": float(target_avg.get("line_length", np.nan)),
        "comparison_line_length": float(comp_avg.get("line_length", np.nan)),
    },
    "top_suspicious_channels": suspicious_top[["channel", "suspicious_score"]].to_dict(orient="records") if len(suspicious_top) > 0 else [],
    "run_consistency": target_mode_df[["run_rank", "std", "hf_ratio", "line_length"]].to_dict(orient="records") if len(target_mode_df) > 0 else [],
}

save_json(summary, ARTIFACT_DIR / "summary.json", enabled=CONFIG["save_artifacts"])

manifest_path = ARTIFACT_DIR / "artifact_manifest.json"
save_json({"run_id": RUN_ID, "artifacts": ARTIFACT_MANIFEST}, manifest_path, enabled=CONFIG["save_artifacts"])

print("\n" + "=" * 70)
print("DIAGNOSTICS SUMMARY")
print("=" * 70)
print(f"Run ID:                    {RUN_ID}")
print(f"Artifact path:             {ARTIFACT_DIR}")
print(f"Target subject:            {CONFIG['target_subject']}")
print(f"Comparison subjects:       {CONFIG['comparison_subjects']}")
print(f"Extra subjects:            {CONFIG['extra_subjects']}")
print(
    "Preprocessing:             "
    f"ref={CONFIG['set_eeg_reference']} | uV={CONFIG['convert_to_uV']} | "
    f"bandpass={CONFIG['bandpass_low']}-{CONFIG['bandpass_high']} Hz | sfreq={CONFIG['sfreq']}"
)
print(f"Window shape:              {window_shape_reference}")
print(f"Total windows:             {len(window_metrics_df)}")
print(f"Outlier windows:           {int(window_metrics_df['outlier_flag'].sum()) if len(window_metrics_df) else 0}")
print(f"Saved artifacts:           {len(ARTIFACT_MANIFEST)} files")
print(f"Manifest path:             {manifest_path}")

if _LOG_FH is not None:
    _LOG_FH.flush()

if CONFIG["save_logs"]:
    builtins.print = _ORIGINAL_PRINT
    _LOG_FH.close()